In [146]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import altair as alt
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import kendalltau

In [147]:
df2 = pd.read_csv('../data/simpsons_episodes_cleaned.csv')


# Q1 How have the ratings evolved over time?

In [148]:
line_chart = alt.Chart(df2).mark_line().encode(
    x=alt.X('original_air_date:T', title='Original Air Date'),
    y=alt.Y('imdb_rating:Q', title='IMDB Rating')
).properties(
    title='IMDB Rating Over Time'
)

season_finals = df2[df2['is_last_episode'] == 1].copy()
limit_bounds = alt.Chart(season_finals).mark_rule(color='black', strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X('original_air_date:T'),
    tooltip=[alt.Tooltip('season:N', title='Season')]
 )

season_ranges = (
    df2.groupby('season', as_index=False)
       .agg(
           start_date=('original_air_date', 'min'),
           end_date=('original_air_date', 'max')
       )
)
season_ranges['end_date'] = pd.to_datetime(season_ranges['end_date'])
season_ranges['start_date'] = pd.to_datetime(season_ranges['start_date'])
season_ranges['mid_date'] = season_ranges['start_date'] + (season_ranges['end_date'] - season_ranges['start_date']) / 3
season_ranges['label_y'] = df2['imdb_rating'].max() * 1.03
season_ranges['season_label'] = 'S' + season_ranges['season'].astype(str)

season_labels = alt.Chart(season_ranges).mark_text(
    dy=-4,
    color='black',
    fontSize=11
).encode(
    x=alt.X('mid_date:T'),
    y=alt.Y('label_y:Q'),
    text=alt.Text('season_label:N', title='Season')
)

chart = line_chart + limit_bounds + season_labels
chart.properties(width=1500, height=400)

alt.LayerChart(...)

In [149]:
df2

,season,number_in_season,number_in_series,original_air_date,day_aired,imdb_rating,us_viewers_in_millions,title,is_last_episode
0,1,10,10,1990-03-25,Sunday,7.4,30.30,Homer's Night Out,0
1,1,12,12,1990-04-29,Sunday,8.3,30.40,Krusty Gets Busted,0
2,2,1,14,1990-10-11,Thursday,8.2,33.60,"Bart Gets an ""F""",0
3,2,4,17,1990-11-01,Thursday,8.1,26.10,Two Cars in Every Garage and Three Eyes on Eve...,0
4,2,6,19,1990-11-15,Thursday,8.0,25.40,Dead Putting Society,0
...,...,...,...,...,...,...,...,...,...
591,23,14,500,2012-02-19,Sunday,7.0,5.77,At Long Last Leave,0
592,23,16,502,2012-03-11,Sunday,7.3,4.97,How I Wet Your Mother,0
593,24,3,511,2012-11-04,Sunday,6.9,5.65,Adventures in Baby-Getting,0
594,25,12,542,2014-03-09,Sunday,6.4,2.69,Diggs,0


In [150]:
import altair as alt


# Configure common options. We specify the aggregation
# as a transform here so we can reuse it in both layers.

# Configure heatmap
heatmap = alt.Chart(df2).mark_rect().encode(
    x = alt.X('season:N',axis=alt.Axis(orient='top',labelAngle=0)),
    y = alt.Y('number_in_season:N'),
    color = alt.Color('imdb_rating:Q',
        scale = alt.Scale(range=['red', 'yellow', 'green'])))


# Configure text
text = alt.Chart(df2).mark_text(baseline='middle').encode(
    x = alt.X('season:N'),
    y = alt.Y('number_in_season:N'),
    text = alt.Text('imdb_rating:Q', format=".1f")
)

# Draw the chart
(heatmap + text).properties(width=600, height=400)

alt.LayerChart(...)

# Q2 How have the viewers evolved over time

In [151]:
df2['season_dec'] = df2['season'] + (df2['number_in_season'] / df2.groupby('season')['number_in_season'].transform('max')) 

In [154]:
df2

,season,number_in_season,number_in_series,original_air_date,day_aired,imdb_rating,us_viewers_in_millions,title,is_last_episode,season_dec
0,1,10,10,1990-03-25,Sunday,7.4,30.30,Homer's Night Out,0,1.769231
1,1,12,12,1990-04-29,Sunday,8.3,30.40,Krusty Gets Busted,0,1.923077
2,2,1,14,1990-10-11,Thursday,8.2,33.60,"Bart Gets an ""F""",0,2.045455
3,2,4,17,1990-11-01,Thursday,8.1,26.10,Two Cars in Every Garage and Three Eyes on Eve...,0,2.181818
4,2,6,19,1990-11-15,Thursday,8.0,25.40,Dead Putting Society,0,2.272727
...,...,...,...,...,...,...,...,...,...,...
591,23,14,500,2012-02-19,Sunday,7.0,5.77,At Long Last Leave,0,23.636364
592,23,16,502,2012-03-11,Sunday,7.3,4.97,How I Wet Your Mother,0,23.727273
593,24,3,511,2012-11-04,Sunday,6.9,5.65,Adventures in Baby-Getting,0,24.136364
594,25,12,542,2014-03-09,Sunday,6.4,2.69,Diggs,0,25.545455


In [163]:
df2['season'] = pd.Categorical(df2['season'], ordered=True)

line = alt.Chart(df2).transform_aggregate(
    mean_viewers='mean(us_viewers_in_millions)',
    groupby=['season']
).mark_line().encode(
    x=alt.X('season:Q', title='Season', axis=alt.Axis(tickCount=26),scale=alt.Scale(domain=[1, 28])),
    y=alt.Y('mean_viewers:Q', title='Average US Viewers (Millions)'),
    tooltip=[
        alt.Tooltip('season:O', title='Season'),
        alt.Tooltip('mean_viewers:Q', title='Average US Viewers (Millions)', format='.2f')
    ]
).properties(
    title='Average US Viewers by Season'
)


point = alt.Chart(df2).mark_circle(size=50, opacity=0.5, color='orange').encode(
    x=alt.X('season_dec:Q', title='Season',scale = alt.Scale(domain=[1, 28])),
    y=alt.Y('us_viewers_in_millions:Q', title='US Viewers (Millions)'),
    tooltip=[
        alt.Tooltip('season:O', title='Season'),
        alt.Tooltip('us_viewers_in_millions:Q', title='US Viewers (Millions)', format='.2f')
    ]
)

# resolve axis 
chart = line + point
chart.properties(width=800, height=400)

alt.LayerChart(...)

In [95]:
line_chart = alt.Chart(df2).mark_line().encode(
    x=alt.X('original_air_date:T', title='Original Air Date'),
    y=alt.Y('us_viewers_in_millions:Q', title='US Viewers (Millions)')
).properties(
    title='US Viewers Over Time'
)

season_finals = df2[df2['is_last_episode'] == 1].copy()
limit_bounds = alt.Chart(season_finals).mark_rule(color='black', strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X('original_air_date:T'),
    tooltip=[alt.Tooltip('season:N', title='Season')]
 )

# Midpoint label (season number) between start and end date of each season
season_ranges = (
    df2.groupby('season', as_index=False)
       .agg(
           start_date=('original_air_date', 'min'),
           end_date=('original_air_date', 'max')
       )
)

season_ranges['end_date'] = pd.to_datetime(season_ranges['end_date'])
season_ranges['start_date'] = pd.to_datetime(season_ranges['start_date'])
season_ranges['mid_date'] = season_ranges['start_date'] + (season_ranges['end_date'] - season_ranges['start_date']) / 3
season_ranges['label_y'] = df2['us_viewers_in_millions'].max() * 1.03
season_ranges['season_label'] = 'S' + season_ranges['season'].astype(str)

season_labels = alt.Chart(season_ranges).mark_text(
    dy=-4,
    color='black',
    fontSize=11
).encode(
    x=alt.X('mid_date:T'),
    y=alt.Y('label_y:Q'),
    text=alt.Text('season_label:N', title='Season')
)

chart = line_chart + limit_bounds + season_labels
chart.properties(width=1200, height=400)

alt.LayerChart(...)

# Q3 -> Is there a correlation between the gradings and the viewers?

We use the nonparametric version of the peerson coeficient since the two varaibles do not share the same sclae

In [53]:
# kendall correlation between season and imdb rating
correlation_kendall, _ = kendalltau(df2['us_viewers_in_millions'], df2['imdb_rating'])
correlation_kendall.round(4)

0.4703

In [54]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df2['imdb_rating_normalized'] = scaler.fit_transform(df2[['imdb_rating']])
df2['us_viewers_in_millions_normalized'] = scaler.fit_transform(df2[['us_viewers_in_millions']])
from scipy.stats import pearsonr
correlation_pearson, _ = pearsonr(df2['us_viewers_in_millions_normalized'], df2['imdb_rating_normalized'])
correlation_pearson.round(4)


0.6216

In [55]:
alt.Chart(df2).mark_point().encode(
    x=alt.X('us_viewers_in_millions:Q', title='US Viewers (Millions)'),
    y=alt.Y('imdb_rating:Q', title='IMDB Rating'),
    tooltip= ['season', 'number_in_season']
).properties(
    title='IMDB Rating vs US Viewers with Season and Episode Tooltips'
)

alt.Chart(...)

In [56]:
alt.Chart(df2).mark_point().encode(
    x=alt.X('us_viewers_in_millions_normalized:Q', title='US Viewers (Millions)'),
    y=alt.Y('imdb_rating_normalized:Q', title='IMDB Rating'),
    tooltip= ['season', 'number_in_season']
).properties(
    title='IMDB Rating vs US Viewers with Season and Episode Tooltips'
)

alt.Chart(...)

In [57]:
base = alt.Chart(df2).encode(
    alt.X('original_air_date:T', axis=alt.Axis(title=None))
)

rating = base.mark_line(stroke='#5276A7', interpolate='monotone').encode(
    alt.Y('imdb_rating:Q', axis=alt.Axis(title='IMDB Rating', titleColor='#5276A7'))
)
viewers = base.mark_line(stroke='#D62728', interpolate='monotone').encode(
    alt.Y('us_viewers_in_millions:Q', axis=alt.Axis(title='US Viewers (Millions)', titleColor='#D62728'))
)

season_labels = alt.Chart(season_ranges).mark_text(
    dy=-4,
    color='black',
    fontSize=11
).encode(
    x=alt.X('mid_date:T'),
    y=alt.Y('label_y:Q',title=''),
    text=alt.Text('season_label:N', title='Season')
)

limit_bounds = alt.Chart(season_finals).mark_rule(color='black', strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X('original_air_date:T'),
    tooltip=[alt.Tooltip('season:N', title='Season')]
 )



alt.layer(rating, viewers, season_labels, limit_bounds).resolve_scale(
    y = 'independent'
).properties(width=1200, height=400, title='IMDB Rating and US Viewers Over Time')

alt.LayerChart(...)

# Q4 Are the number of viewers for the episodes related to the weekday they were aired?

In [58]:
valid_airing = df2[df2['day_aired'].isin(['Sunday','Thursday'])] 

In [59]:


jitter = alt.Chart(valid_airing, title='Normally distributed jitter').mark_circle(size=8).encode(
    y="day_aired:N",
    x="imdb_rating:Q",
    yOffset="jitter:Q",
).transform_calculate( jitter='random()'
).encode(
    y=alt.Y('day_aired:N')
).properties(
    title='Uniformly distributed jitter',
    height=100,
    width=300
)

jitter





alt.Chart(...)

In [60]:
valid_airing = df2[df2['day_aired'].isin(['Sunday', 'Thursday'])]

jitter = (
    alt.Chart(valid_airing)
    .transform_calculate(jitter="(random() - 0.5) * 20")
    .mark_circle(size=25, opacity=0.5)
    .encode(
        x=alt.X("day_aired:N"),
        y=alt.Y("imdb_rating:Q"),
        xOffset="jitter:Q"
    )
    .properties(width=220, height=300, title="IMDb rating by day with horizontal jitter")
)

jitter

alt.Chart(...)

In [62]:
line_chart = alt.Chart(valid_airing).mark_line().encode(
    x=alt.X('original_air_date:T', title='Original Air Date'),
    y=alt.Y('us_viewers_in_millions:Q', title='US Viewers (Millions)'),
    color = 'day_aired'
).properties(
    title='US Viewers Over Time'
)

season_finals = df2[df2['is_last_episode'] == 1].copy()
limit_bounds = alt.Chart(season_finals).mark_rule(color='black', strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X('original_air_date:T'),
    tooltip=[alt.Tooltip('season:N', title='Season')]
 )

# Midpoint label (season number) between start and end date of each season
season_ranges = (
    df2.groupby('season', as_index=False)
       .agg(
           start_date=('original_air_date', 'min'),
           end_date=('original_air_date', 'max')
       )
)
season_ranges['end_date'] = pd.to_datetime(season_ranges['end_date'])
season_ranges['start_date'] = pd.to_datetime(season_ranges['start_date'])
season_ranges['mid_date'] = season_ranges['start_date'] + (season_ranges['end_date'] - season_ranges['start_date']) / 3
season_ranges['label_y'] = df2['us_viewers_in_millions'].max() * 1.03
season_ranges['season_label'] = 'S' + season_ranges['season'].astype(str)

season_labels = alt.Chart(season_ranges).mark_text(
    dy=-4,
    color='black',
    fontSize=11
).encode(
    x=alt.X('mid_date:T'),
    y=alt.Y('label_y:Q'),
    text=alt.Text('season_label:N', title='Season')
)

chart = line_chart + limit_bounds + season_labels
chart.properties(width=1200, height=400)

alt.LayerChart(...)

it looked like the ones airing on a thurday had better ratings, but with this second plot we see that they only aired on first season, so the comparative is not fair

# Q5 Do the seasons’ number of viewers present any relevant pattern?

Look for seasonality

In [ ]:
df2

In [ ]:
#df2['dif_vs_season_avg'] = df2['us_viewers_in_millions'] - df2.groupby('season')['us_viewers_in_millions'].transform('median')
df2['dif_vs_season_avg'] = df2['us_viewers_in_millions'] - df2.groupby('season')['us_viewers_in_millions'].transform('mean')

In [ ]:
alt.Chart(df2).mark_point().encode(
    x=alt.X('number_in_season:Q', title='number_in_season'),
    y=alt.Y('dif_vs_season_avg:Q', title='Difference vs Season Average'),
    tooltip= ['season', 'number_in_season']
).properties(
    title='IMDB Rating vs US Viewers with Season and Episode Tooltips'
)

sembla q els ultims es beuen menys ja que hi ha mñes diferència vs la mitjana de la temporada, però no es veu rés clar del tot

In [ ]:
line_chart = alt.Chart(df2).mark_line().encode(
    x=alt.X('original_air_date:T', title='Original Air Date'),
    y=alt.Y('dif_vs_season_avg:Q', title='US Viewers (Millions)')
).properties(
    title='Average vs Season Average US Viewers Over Time'
)

season_finals = df2[df2['is_last_episode'] == 1].copy()
limit_bounds = alt.Chart(season_finals).mark_rule(color='black', strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X('original_air_date:T'),
    tooltip=[alt.Tooltip('season:N', title='Season')]
 )

# Midpoint label (season number) between start and end date of each season
season_ranges = (
    df2.groupby('season', as_index=False)
       .agg(
           start_date=('original_air_date', 'min'),
           end_date=('original_air_date', 'max')
       )
)
season_ranges['mid_date'] = season_ranges['start_date'] + (season_ranges['end_date'] - season_ranges['start_date']) / 3
season_ranges['label_y'] = df2['dif_vs_season_avg'].max() * 1.03
season_ranges['season_label'] = 'S' + season_ranges['season'].astype(str)

season_labels = alt.Chart(season_ranges).mark_text(
    dy=-4,
    color='black',
    fontSize=11
).encode(
    x=alt.X('mid_date:T'),
    y=alt.Y('label_y:Q'),
    text=alt.Text('season_label:N', title='Season')
)

chart = line_chart + limit_bounds + season_labels
chart.properties(width=1500, height=400)

In [ ]:
df2

In [ ]:
avg_episode_in_season = df2.groupby(['number_in_season'], as_index=False)['us_viewers_in_millions'].mean()
avg_episode_in_season

#avg_episode_in_season = avg_episode_in_season.merge(df2[['number_in_season','original_air_date']], on='number_in_season', how='left')

In [ ]:
alt.Chart(avg_episode_in_season).mark_bar().encode(
    x=alt.X('number_in_season:Q', title='Episode Number in Season'),
    y=alt.Y('us_viewers_in_millions:Q', title='Average US Viewers (Millions)')
).properties(
    title='Average US Viewers by Episode Number in Season'
)

apilades

In [ ]:
# US viewers per episode number, faceted vertically by season with shared X axis
(
    alt.Chart(df2)
    .mark_line()
    .encode(
        x=alt.X("number_in_season:Q", title="Episode Number in Season"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (Millions)")
    )
    .facet(
        row=alt.Row("season:N", title="Season")
    )
    .resolve_scale(
        x="shared",
        y="independent"
    )
    .properties(
        title="US Viewers by Episode Number in Season, Faceted by Season"
    )
)

grid

In [64]:
# US viewers per episode number in a 5-column grid (last row contains remaining seasons)
(
    alt.Chart(df2)
    .mark_line()
    .encode(
        x=alt.X("number_in_season:Q", title="Episode Number in Season"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (Millions)")
    )
    .facet(
        facet=alt.Facet("season:N", title="Season"),
        columns=5
    )
    .resolve_scale(
        x="shared",
        y="shared"
    )
    .properties(
        title="US Viewers by Episode Number in Season (5x5 Grid + Remaining Seasons Below)",
    )
)



alt.FacetChart(...)

In [ ]:
# Q2
alt.Chart(df).mark_line().encode(
    x='original_air_date',
    y='average(us_viewers_in_millions)'
)

In [ ]:
# Q3
alt.Chart(df).mark_circle().encode(
    x='us_viewers_in_millions',
    y='imdb_rating'
)

In [ ]:
# Q4
# Day Rating Distribution
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='average(imdb_rating)'
)